[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C11_RAG_Retrieval_Course/01_embeddings_semantic_search/01_embeddings_semantic_search.ipynb)

# 01 · 嵌入与语义搜索（纯 numpy/pandas）

目标：把 **embedding 向量、点积/余弦/L2 归一化、玩具 embedding、BM25、语义 vs 词法、混合检索（分数加权 + RRF）、向量化检索** 全部从零实现，并用 `assert` 验证。

路线：余弦/归一化 → 玩具 embedding → 语义检索 top-k → BM25 从零 → 语义 vs 词法对比 → 混合检索 → ✏️ 练习 → 📖 答案 → 🧪 真实数据(SQuAD)胶囊。

> 心智模型：**embedding 把『语义近』变成『向量近』；检索 = 归一化后的一次矩阵乘 + 取 top-k**。我们冻结『向量怎么学』，专练『有了向量之后』的整条流水线。

## 1 · 相似度：点积、余弦、L2 归一化

点积 `a·b = Σ aᵢbᵢ` 混了方向与模长；余弦 `a·b/(‖a‖‖b‖)` 只看方向，是语义检索的主力。

**关键事实**：L2 归一化后，点积 = 余弦；且单位向量上 `‖a-b‖² = 2 - 2cos`，所以**最大点积 = 最大余弦 = 最小欧氏距离**（三者排序等价）。下面逐一验证。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def l2_normalize(M, eps=1e-12):
    M = np.asarray(M, dtype=float)
    return M / (np.linalg.norm(M, axis=-1, keepdims=True) + eps)

a = np.array([3.0, 4.0]); b = np.array([4.0, 3.0])
print('点积 a·b      =', a @ b)
print('余弦 cos(a,b) =', round(cosine(a, b), 4))

# 归一化后点积 == 余弦
an, bn = l2_normalize(a[None])[0], l2_normalize(b[None])[0]
assert abs(an @ bn - cosine(a, b)) < 1e-12, '归一化后点积应等于余弦'
assert abs(np.linalg.norm(an) - 1.0) < 1e-12, '归一化后模长应为 1'
# 单位向量上 ‖a-b‖² == 2 - 2cos
lhs = np.sum((an - bn) ** 2)
rhs = 2 - 2 * (an @ bn)
assert abs(lhs - rhs) < 1e-12, '单位向量上 ‖a-b‖²=2-2cos'
print('✅ 归一化后：点积=余弦；且 ‖a-b‖²=2-2cos -> 最大点积=最小欧氏距离（排序等价）')

## 2 · 归一化的陷阱：零向量

归一化要除以范数，**零向量会除出 nan/inf**。这是检索代码最常见的低级 bug。正确实现永远写 `v / (norm + eps)`。下面验证带 eps 的实现对零向量安全、对正常向量无损。

In [ ]:
M = np.array([
    [3.0, 4.0],     # 正常向量, 范数 5
    [0.0, 0.0],     # 零向量！
    [1.0, 0.0],
])
N = l2_normalize(M)
print('归一化结果:\n', np.round(N, 4))
assert not np.isnan(N).any() and not np.isinf(N).any(), '带 eps 不应产生 nan/inf'
assert abs(np.linalg.norm(N[0]) - 1.0) < 1e-9, '正常向量归一化后模长 1'
assert np.allclose(N[1], 0.0), '零向量归一化后仍≈0（被 eps 保护，不爆炸）'
print('✅ 带 eps 的归一化：正常向量无损、零向量安全（不产生 nan）')

## 3 · 玩具 embedding + 语义检索 top-k

没有神经网络，也能**确定性地**造有语义结构的向量：**哈希词袋**——每个词哈希到若干维 +1，文本向量 = 其词向量之和。共享词多的文本，向量自然接近。

然后做语义检索：归一化库向量，查询向量与库做一次矩阵乘得余弦，取 top-k。

In [ ]:
def hash_embed(text, dim=64, seed=0):
    '''确定性哈希词袋：把每个词哈希到 dim 维里的一个坐标 +1。零训练。'''
    v = np.zeros(dim)
    for tok in text.lower().split():
        h = (hash((seed, tok)) % dim)
        v[h] += 1.0
    return v

# 注意：Python 的内置 hash 对 str 默认加盐(每次运行不同)，为可复现这里用一个稳定哈希
def stable_hash(s):
    h = 1469598103934665603
    for ch in s:
        h = (h ^ ord(ch)) * 1099511628211 % (2**64)
    return h

def embed(text, dim=64):
    v = np.zeros(dim)
    for tok in text.lower().split():
        v[stable_hash(tok) % dim] += 1.0
    return v

corpus = [
    'the cat sat on the mat',
    'a dog ran in the park',
    'cats and dogs are common pets',
    'the stock market fell sharply today',
    'investors sold stocks amid market fear',
]
D = l2_normalize(np.array([embed(t) for t in corpus]))

def search(query, D, corpus, k=3):
    q = l2_normalize(embed(query)[None])[0]
    scores = D @ q                       # 一次矩阵乘 = 全部余弦
    idx = np.argsort(-scores)[:k]
    return [(corpus[i], round(float(scores[i]), 3)) for i in idx]

hits = search('pets cat dog', D, corpus, k=3)
for doc, sc in hits:
    print(f'{sc:+.3f}  {doc}')
# 与“宠物猫狗”最相关的应是前 3 条动物文档，而非股市文档
top_docs = [d for d, _ in hits]
assert 'the stock market fell sharply today' not in top_docs, '股市文档不该进 top-3'
assert any('cat' in d or 'dog' in d or 'pets' in d for d in top_docs)
print('✅ 语义检索把动物相关文档排在前面，股市文档被排除')

## 4 · BM25：从零实现词法检索之王

BM25 给查询每个词的贡献 = `IDF(t) × 饱和词频`，再求和：

$$\text{BM25}(q,d)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{tf(t,d)(k_1+1)}{tf(t,d)+k_1(1-b+b\,|d|/\text{avgdl})}$$

三件套：**IDF**（罕见词权重高）+ **词频饱和**（多次出现边际递减）+ **长度归一化**（别偏向长文档）。

In [ ]:
import math
from collections import Counter

class BM25:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [d.lower().split() for d in corpus]
        self.N = len(self.docs)
        self.doc_len = [len(d) for d in self.docs]
        self.avgdl = sum(self.doc_len) / self.N
        # df: 含每个词的文档数
        df = Counter()
        for d in self.docs:
            for t in set(d):
                df[t] += 1
        # idf（加 0.5 平滑，保证非负）
        self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
        self.tf = [Counter(d) for d in self.docs]

    def score(self, query, i):
        s = 0.0
        dl = self.doc_len[i]
        for t in query.lower().split():
            if t not in self.idf:
                continue
            tf = self.tf[i][t]
            denom = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            s += self.idf[t] * tf * (self.k1 + 1) / denom
        return s

    def search(self, query, k=3):
        scores = np.array([self.score(query, i) for i in range(self.N)])
        idx = np.argsort(-scores)[:k]
        return idx, scores

bm = BM25(corpus)
idx, scores = bm.search('market stocks', k=3)
print('查询 “market stocks” 的 BM25 top-3:')
for i in idx:
    print(f'  {scores[i]:.3f}  {corpus[i]}')
# 含 market/stocks 的两条股市文档应排最前
assert 'market' in corpus[idx[0]] or 'stock' in corpus[idx[0]]
# 罕见词 IDF 应高于常见词 the
assert bm.idf['market'] > bm.idf['the'], 'market 比 the 罕见 -> IDF 更高'
print('✅ BM25 从零实现正确：罕见词 IDF 更高，含查询词的文档排前')

## 5 · 语义 vs 词法：盲区互补

构造一个**专门暴露盲区**的小场景：
- 查询含一个**罕见精确词**（如型号 `xj9`）→ 词法强、语义弱（玩具 embedding 编不准罕见词）。
- 查询需要**同义匹配**（共享词指向同一主题）→ 语义能靠共享词找到，词法也行但更脆。

我们对比两路对同一查询的命中，看它们漏的不是同一批。

In [ ]:
corpus2 = [
    'the xj9 turbocharger improves engine power',   # 含精确型号 xj9
    'this engine has great horsepower and torque',  # 同主题(引擎)但无 xj9
    'a recipe for chocolate cake with frosting',    # 无关
]
D2 = l2_normalize(np.array([embed(t) for t in corpus2]))
bm2 = BM25(corpus2)

q = 'xj9 engine'
# 词法 top-1
lex_idx, _ = bm2.search(q, k=3)
# 语义 top-1
qv = l2_normalize(embed(q)[None])[0]
sem_idx = np.argsort(-(D2 @ qv))[:3]

print('词法 BM25 排序:', [corpus2[i][:30] for i in lex_idx])
print('语义     排序:', [corpus2[i][:30] for i in sem_idx])
# 词法把含精确词 xj9 的文档排第一（精确匹配强）
assert 'xj9' in corpus2[lex_idx[0]], '词法应精确命中 xj9'
# 两路都应把无关的蛋糕文档排在最后
assert 'cake' in corpus2[lex_idx[-1]] and 'cake' in corpus2[sem_idx[-1]]
print('✅ 词法精确命中罕见型号 xj9；两路都能排除无关文档，但强项不同 -> 该融合')

## 6 · 混合检索：分数加权 vs RRF

**分数加权**：两路分数各自 min-max 归一化后加权求和（难点=归一化）。
**RRF（倒数排名融合）**：只用名次，`融合分 = Σ 1/(k_const + rank)`，无需归一化、异常稳健。

下面实现两者，并验证融合结果的合理性。

In [ ]:
def minmax(x):
    x = np.asarray(x, dtype=float)
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo + 1e-12)

def hybrid_weighted(sem_scores, lex_scores, alpha=0.5):
    '''两路分数各自 min-max 归一化后加权。'''
    return alpha * minmax(sem_scores) + (1 - alpha) * minmax(lex_scores)

def rrf(rank_lists, k_const=60):
    '''rank_lists: 多路的排序结果(每路是文档下标按相关性降序的列表)。返回每个文档的 RRF 分。'''
    n = max(max(rl) for rl in rank_lists) + 1
    fused = np.zeros(n)
    for rl in rank_lists:
        for rank, doc in enumerate(rl):      # rank 从 0 开始
            fused[doc] += 1.0 / (k_const + rank + 1)
    return fused

# 用第 5 节的场景：语义分、词法分
sem_scores = D2 @ qv
lex_scores = np.array([bm2.score(q, i) for i in range(len(corpus2))])

w = hybrid_weighted(sem_scores, lex_scores, alpha=0.5)
sem_rank = np.argsort(-sem_scores).tolist()
lex_rank = np.argsort(-lex_scores).tolist()
r = rrf([sem_rank, lex_rank])

print('加权融合分:', np.round(w, 3), '-> top:', int(np.argmax(w)))
print('RRF  融合分:', np.round(r, 4), '-> top:', int(np.argmax(r)))
# 两种融合都该把无关蛋糕文档(下标2)排最后
assert np.argmin(w) == 2 and np.argmin(r) == 2, '蛋糕文档应融合分最低'
# RRF 对“在两路都靠前”的文档给高分
assert r[0] > r[2], 'xj9 文档(两路都不差)应高于无关文档'
print('✅ 两种融合都正确排除无关文档；RRF 无需归一化即稳健融合多路')

---
## ✏️ 练习 1：从零实现余弦相似度（批量）

实现 `cosine_matrix(Q, D)`：`Q` 是 `(m,d)` 查询矩阵，`D` 是 `(n,d)` 库矩阵，返回 `(m,n)` 的余弦相似度矩阵。**不要用循环**，用矩阵乘 + 归一化。

In [ ]:
def cosine_matrix(Q, D, eps=1e-12):
    # TODO: 分别 L2 归一化 Q 和 D（按行），再 Qn @ Dn.T 得 (m,n) 余弦矩阵
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Q = rng.standard_normal((4, 8)); Dm = rng.standard_normal((6, 8))
S = cosine_matrix(Q, Dm)
assert S.shape == (4, 6)
assert np.all(S <= 1.0 + 1e-9) and np.all(S >= -1.0 - 1e-9), '余弦应在[-1,1]'
# 与逐对手算一致
for i in range(4):
    for j in range(6):
        c_ij = Q[i] @ Dm[j] / (np.linalg.norm(Q[i]) * np.linalg.norm(Dm[j]))
        assert abs(S[i, j] - c_ij) < 1e-9
# 自己与自己余弦=1
assert np.allclose(np.diag(cosine_matrix(Dm, Dm)), 1.0, atol=1e-9)
print('✅ 练习 1 通过：向量化余弦矩阵正确')

## ✏️ 练习 2：L2 归一化让三种相似度排序等价

实现 `topk_by(metric, q, D, k)`，`metric ∈ {'dot','cosine','neg_l2'}`：分别用点积/余弦/负欧氏距离给库排序取 top-k。

验证：**对已 L2 归一化的库**，三种 metric 给出**完全相同**的 top-k 顺序。

In [ ]:
def topk_by(metric, q, D, k=5):
    # TODO: 按 metric 计算每个库向量的分数(越大越相关)，返回 top-k 下标
    #   'dot'    -> D @ q
    #   'cosine' -> 归一化后的点积
    #   'neg_l2' -> -‖D-q‖（取负，使越大越近）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Dn = l2_normalize(rng.standard_normal((50, 16)))
qn = l2_normalize(rng.standard_normal(16)[None])[0]   # 查询也归一化
a = topk_by('dot', qn, Dn, k=5)
b = topk_by('cosine', qn, Dn, k=5)
cc = topk_by('neg_l2', qn, Dn, k=5)
assert list(a) == list(b) == list(cc), '归一化后三种相似度的 top-k 应完全一致'
print('top-5 (dot)   :', list(a))
print('top-5 (cosine):', list(b))
print('top-5 (neg_l2):', list(cc))
print('✅ 练习 2 通过：归一化后 点积/余弦/欧氏 排序等价')

## ✏️ 练习 3：BM25 打分

复用第 4 节的 `BM25` 类。实现 `bm25_top1(corpus, query)`：返回 BM25 最相关文档的**下标**。

再验证一条 BM25 的关键性质：**词频饱和**——同一个词出现更多次，分数增加但是**次线性**的（边际递减）。

In [ ]:
def bm25_top1(corpus, query, k1=1.5, b=0.75):
    # TODO: 用 BM25 类，返回得分最高文档的下标(int)
    raise NotImplementedError

def saturation_gain(k1=1.5):
    '''返回 (词频1→2 的分数增量, 词频5→6 的分数增量)，用于验证饱和(后者应更小)。'''
    # TODO: 用单文档、忽略长度归一化(b=0, dl=avgdl)，f(tf)=tf*(k1+1)/(tf+k1)
    #   返回 (f(2)-f(1), f(6)-f(5))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
cps = ['apple banana cherry', 'banana banana banana split', 'unrelated text here']
t1 = bm25_top1(cps, 'banana')
assert t1 == 1, '“banana”出现3次的文档应 BM25 最高'
g_low, g_high = saturation_gain()
assert g_low > g_high > 0, '词频饱和：1→2 的增量应大于 5→6 的增量(次线性)'
print(f'词频 1→2 增量={g_low:.4f}  >  5→6 增量={g_high:.4f}')
print('✅ 练习 3 通过：BM25 打分正确且体现词频饱和')

## ✏️ 练习 4：RRF 混合检索

实现 `reciprocal_rank_fusion(rank_lists, k_const=60)`：输入多路排序(每路是文档下标的降序列表)，返回按 RRF 融合分**降序**排好的文档下标列表。

验证：一个在两路都排第一的文档，融合后必排第一；只在一路出现的文档不应超过它。

In [ ]:
def reciprocal_rank_fusion(rank_lists, k_const=60):
    # TODO: 对每路每个文档累加 1/(k_const + rank+1)，返回按融合分降序的下标列表
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 路1: 文档2最相关; 路2: 文档2也最相关 -> 融合后2必第一
fused = reciprocal_rank_fusion([[2, 0, 1, 3], [2, 1, 0, 3]])
assert fused[0] == 2, '两路都第一的文档应融合第一'
assert fused[-1] == 3, '两路都垫底的文档应融合垫底'
# 一致性：单路时 RRF 排序 == 原排序
single = reciprocal_rank_fusion([[5, 3, 1, 0]])
assert single == [5, 3, 1, 0], '单路 RRF 应还原原排序'
print('RRF 融合排序:', fused)
print('✅ 练习 4 通过：RRF 正确融合多路排序')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cosine_matrix(Q, D, eps=1e-12):
    Qn = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + eps)
    Dn = D / (np.linalg.norm(D, axis=1, keepdims=True) + eps)
    return Qn @ Dn.T

In [ ]:
# 练习 2 参考答案
def topk_by(metric, q, D, k=5):
    if metric == 'dot':
        s = D @ q
    elif metric == 'cosine':
        Dn = D / (np.linalg.norm(D, axis=1, keepdims=True) + 1e-12)
        qn = q / (np.linalg.norm(q) + 1e-12)
        s = Dn @ qn
    elif metric == 'neg_l2':
        s = -np.linalg.norm(D - q, axis=1)
    else:
        raise ValueError(metric)
    return np.argsort(-s)[:k]

In [ ]:
# 练习 3 参考答案
def bm25_top1(corpus, query, k1=1.5, b=0.75):
    bm = BM25(corpus, k1=k1, b=b)
    idx, _ = bm.search(query, k=1)
    return int(idx[0])

def saturation_gain(k1=1.5):
    f = lambda tf: tf * (k1 + 1) / (tf + k1)
    return (f(2) - f(1), f(6) - f(5))

In [ ]:
# 练习 4 参考答案
def reciprocal_rank_fusion(rank_lists, k_const=60):
    from collections import defaultdict
    score = defaultdict(float)
    for rl in rank_lists:
        for rank, doc in enumerate(rl):
            score[doc] += 1.0 / (k_const + rank + 1)
    return [doc for doc, _ in sorted(score.items(), key=lambda kv: -kv[1])]

---
## 🧪 真实数据胶囊：在 SQuAD 上做语义 vs 词法检索

用真实的 **SQuAD**（Stanford Question Answering Dataset）问答数据：每个问题对应一个真实段落。我们优先**联网下载**，失败则**回退到内置的真实 SQuAD 样本**（逐字摘自数据集），保证离线也能跑、逻辑一致。

任务：把段落当语料、问题当查询，对比玩具语义检索与 BM25 谁能把『含答案的段落』排得更靠前。

In [ ]:
# 优先联网拉真实 SQuAD；失败回退到内置真实样本
def load_squad_samples(n=5):
    try:
        import urllib.request, json
        url = 'https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json'
        with urllib.request.urlopen(url, timeout=5) as f:
            data = json.load(f)
        out = []
        # 每个段落只取 1 个问题 -> 保证 n 个查询对应 n 个【不同】段落(top-1命中才有意义)
        for art in data['data']:
            for para in art['paragraphs']:
                ctx = para['context']
                for qa in para['qas']:
                    if not qa.get('is_impossible', False) and qa['answers']:
                        out.append((qa['question'], ctx, qa['answers'][0]['text']))
                        break          # 本段落只取 1 个问题
                if len(out) >= n: break
            if len(out) >= n: break
        print(f'✅ 联网加载 SQuAD 成功，取 {len(out)} 条(各属不同段落)')
        return out
    except Exception as e:
        print(f'⚠ 联网失败({type(e).__name__})，回退内置真实 SQuAD 样本')
        # 以下为逐字摘自 SQuAD dev 集的真实样本(问题/段落节选/答案)
        return [
            ('In what country is Normandy located?',
             'The Normans were the people who in the 10th and 11th centuries gave their '
             'name to Normandy, a region in France. They were descended from Norse raiders.',
             'France'),
            ('When were the Normans in Normandy?',
             'The Normans were the people who in the 10th and 11th centuries gave their '
             'name to Normandy, a region in France.',
             '10th and 11th centuries'),
            ('What is the capital of France?',
             'Paris is the capital and most populous city of France, situated on the Seine.',
             'Paris'),
            ('What language did the Normans speak?',
             'The Norman dynasty had a major political and cultural impact; they spoke a '
             'language that evolved into Norman French.',
             'Norman French'),
            ('Photosynthesis occurs in which organelle?',
             'In plants, photosynthesis takes place in chloroplasts, which contain the '
             'pigment chlorophyll that captures light energy.',
             'chloroplasts'),
        ]

samples = load_squad_samples(5)
questions = [q for q, _, _ in samples]
contexts  = [c for _, c, _ in samples]
print(f'\n语料: {len(contexts)} 个段落; 查询: {len(questions)} 个问题')
print('例:', questions[0][:60], '...')

**🧪 胶囊练习**：实现 `retrieval_accuracy(questions, contexts, method)`：每个问题 `i` 的正确答案段落就是 `contexts[i]`。用 `method ∈ {'semantic','bm25'}` 检索，返回 **top-1 命中率**（问题 i 检索到的 top-1 恰好是 contexts[i] 的比例）。

In [ ]:
def retrieval_accuracy(questions, contexts, method='semantic'):
    # TODO: 对每个问题 i 检索 contexts，若 top-1 下标==i 记一次命中；返回命中率
    #   semantic: 用 embed + 余弦; bm25: 用 BM25 类
    raise NotImplementedError

In [ ]:
# 自测
n_q = len(questions)
acc_sem = retrieval_accuracy(questions, contexts, 'semantic')
acc_bm  = retrieval_accuracy(questions, contexts, 'bm25')
random_baseline = 1.0 / n_q                 # 瞎猜的命中率
print(f'语料/查询规模: {n_q}')
print(f'语义检索 top-1 命中率: {acc_sem:.2f}')
print(f'BM25   top-1 命中率: {acc_bm:.2f}')
print(f'随机基线       命中率: {random_baseline:.2f}')
assert 0.0 <= acc_sem <= 1.0 and 0.0 <= acc_bm <= 1.0
# 关键且稳健的论断：BM25 把含答案的段落排在 top-1 的概率，远高于瞎猜
assert acc_bm > random_baseline, 'BM25 应显著优于随机基线'
print('✅ 胶囊练习通过：在真实 SQuAD 上两种检索均显著优于随机基线')

In [ ]:
# 📖 胶囊参考答案
def retrieval_accuracy(questions, contexts, method='semantic'):
    hits = 0
    if method == 'semantic':
        D = l2_normalize(np.array([embed(c) for c in contexts]))
        for i, q in enumerate(questions):
            qv = l2_normalize(embed(q)[None])[0]
            top1 = int(np.argmax(D @ qv))
            hits += (top1 == i)
    elif method == 'bm25':
        bm = BM25(contexts)
        for i, q in enumerate(questions):
            idx, _ = bm.search(q, k=1)
            hits += (int(idx[0]) == i)
    return hits / len(questions)

### 小结
- **embedding** 把『语义近』变成『向量近』；**余弦**只看方向，**L2 归一化**后 点积=余弦=最小欧氏距离(排序等价)。
- **玩具 embedding**(哈希词袋)零训练即可驱动检索实验；我们冻结『向量怎么学』，专练『有了向量之后』。
- **BM25** = IDF + 词频饱和 + 长度归一化，是难击败的词法强基线；**别上了 embedding 就扔掉它**。
- **语义 vs 词法** 盲区互补(改写 vs 罕见精确词)；**混合检索**(分数加权/RRF)取长补短，RRF 无需归一化更稳健。
- 检索 = **归一化 + 一次矩阵乘 + 取 top-k**(argsort/argpartition)；这是下一模块『近似最近邻』要加速的对象。

下一站：**模块 02 · 向量检索** —— 当库有上亿条，O(Nd) 的暴力扫描扛不住，怎么用 IVF/HNSW/PQ 又快又准地找近邻。